In [1]:
import pandas as pd

# ==============================================================================
# DATA
# ==============================================================================

#-------------------------
# DATA 2018 - 2022
#-------------------------
df1_20182022 = pd.read_csv('jumlah_tenaga_kesehatan.csv')

df1_20182022

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun
0,1,11,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH TENAGA DOKTER SPESIALIS,52,ORANG,2018
1,1,21,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH TENAGA KETEKNISAN MEDIS,75,ORANG,2018
2,1,31,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH TENAGA DOKTER GIGI,33,ORANG,2018
3,1,41,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH TENAGA PERAWAT,619,ORANG,2018
4,1,51,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018-S2,JUMLAH TENAGA KEFARMASIAN,50,ORANG,2018
...,...,...,...,...,...,...,...,...,...,...,...
2845,190,2846190,35,JAWA TIMUR,3579,KOTA BATU,2022-S2,JUMLAH TENAGA KESEHATAN MASYARAKAT,44,ORANG,2022
2846,190,2847190,35,JAWA TIMUR,3579,KOTA BATU,2022-S2,JUMLAH TENAGA KEFARMASIAN,170,ORANG,2022
2847,190,2848190,35,JAWA TIMUR,3579,KOTA BATU,2022-S2,JUMLAH TENAGA DOKTER SPESIALIS,57,ORANG,2022
2848,190,2849190,35,JAWA TIMUR,3579,KOTA BATU,2022-S2,JUMLAH TENAGA KESEHATAN LINGKUNGAN,11,ORANG,2022


DATA CLEANING

In [2]:
#===============================================================================
# DATA CLEANING 2018 - 2022
#===============================================================================

df1_20182022 = df1_20182022[
    df1_20182022['kategori'].isin([
        'JUMLAH TENAGA DOKTER UMUM',
        'JUMLAH TENAGA PERAWAT',
        'JUMLAH TENAGA BIDAN'
    ])
]

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df1_20182022.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'id',
    'id_index',
    'kode_provinsi',
    'kode_kabupaten_kota',
    'periode_update'
]

df1_20182022[kolom_string] = df1_20182022[kolom_string].astype(str)
df1_20182022.info()

# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df1_20182022['nama_kabupaten_kota'] = (
    df1_20182022['nama_kabupaten_kota']
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
    nama for nama in daftar_kabkot
    if nama not in df1_20182022['nama_kabupaten_kota'].values
]

print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")

# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df1_20182022.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df1_20182022 = df1_20182022.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


# -------------------------
# . CEK MISSING VALUE
# -------------------------
missing_value = df1_20182022.isnull().sum()

print("\n4.Cek Missing Value")
for kolom in df1_20182022.columns:
    indeks_nan = df1_20182022[df1_20182022[kolom].isna()].index.tolist()

    if len(indeks_nan) > 0:
        print(f"{kolom}: {indeks_nan}")
        

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())
    kolom_numerik = df1_20182022.select_dtypes(include='number').columns
    for kolom in kolom_numerik:
        df1_20182022[kolom] = df1_20182022[kolom].fillna(df1_20182022[kolom].median())
    print("Berhasil ditangani")

else:
    print("Tidak ada missing value")



# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df1_20182022.select_dtypes(include='number').columns

print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df1_20182022[kolom].quantile(0.25)
    Q3 = df1_20182022[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df1_20182022[
        (df1_20182022[kolom] < batas_bawah) |
        (df1_20182022[kolom] > batas_atas)
    ]


    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['nama_kabupaten_kota', 'tahun', 'kategori', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
Index: 570 entries, 3 to 2842
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   570 non-null    int64 
 1   id_index             570 non-null    int64 
 2   kode_provinsi        570 non-null    int64 
 3   nama_provinsi        570 non-null    object
 4   kode_kabupaten_kota  570 non-null    int64 
 5   nama_kabupaten_kota  570 non-null    object
 6   periode_update       570 non-null    object
 7   kategori             570 non-null    object
 8   jumlah               570 non-null    int64 
 9   satuan               570 non-null    object
 10  tahun                570 non-null    int64 
dtypes: int64(6), object(5)
memory usage: 53.4+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
Index: 570 entries, 3 to 2842
Data columns (total 11 columns):
 #   Column               Non

C:\Users\CECIL\AppData\Local\Temp\ipykernel_23328\974095669.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1_20182022[kolom_string] = df1_20182022[kolom_string].astype(str)
C:\Users\CECIL\AppData\Local\Temp\ipykernel_23328\974095669.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1_20182022['nama_kabupaten_kota'] = (


,nama_kabupaten_kota,tahun,kategori,jumlah
218,KABUPATEN SIDOARJO,2018,JUMLAH TENAGA PERAWAT,3164
477,KOTA MALANG,2018,JUMLAH TENAGA PERAWAT,2929
545,KOTA SURABAYA,2018,JUMLAH TENAGA PERAWAT,6872
671,KABUPATEN MALANG,2019,JUMLAH TENAGA PERAWAT,2661
793,KABUPATEN SIDOARJO,2019,JUMLAH TENAGA PERAWAT,2856
1036,KOTA MALANG,2019,JUMLAH TENAGA PERAWAT,3043
1115,KOTA SURABAYA,2019,JUMLAH TENAGA PERAWAT,6999
1237,KABUPATEN MALANG,2020,JUMLAH TENAGA PERAWAT,2735
1355,KABUPATEN SIDOARJO,2020,JUMLAH TENAGA PERAWAT,3028
1607,KOTA MALANG,2020,JUMLAH TENAGA PERAWAT,3316



Variabel: tahun
Tidak ada outlier


In [3]:
# -------------------------
# DATA 2023 - 2024
# -------------------------
df2_20232024 = pd.read_csv('jmlh_tng_kshtn_mnrt_kbptn_kt_cl.csv')

df2_20232024

,id,id_kategori,tahun,kabupatenkota,kategori,jumlah
0,1,522,2024,Kabupaten Pacitan,tenaga_kesehatan_dokter,0
1,1,522,2024,Kabupaten Pacitan,jumlah_tenaga_kesehatan_tradisional,3
2,1,522,2024,Kabupaten Pacitan,tenaga_kesehatan_tenaga_kefarmasian,739
3,1,522,2024,Kabupaten Pacitan,tenaga_kesehatan_bidan,3
4,1,522,2024,Kabupaten Pacitan,tenaga_kesehatan_tenaga_kesehatan_lingkungan,155
...,...,...,...,...,...,...
2023,156,522,2018,Jawa Timur,jumlah_tenaga_medis,0
2024,156,522,2018,Jawa Timur,tenaga_kesehatan_tenaga_gizi,1924
2025,156,522,2018,Jawa Timur,jumlah_tenaga_kesehatan_tradisional,0
2026,156,522,2018,Jawa Timur,tenaga_kesehatan_bidan,23085


In [4]:
#===============================================================================
# DATA CLEANING 2023 - 2024
#===============================================================================

# -------------------------
# 1. INFO DATA
# -------------------------
print("\n 1. Info Data:")
df2_20232024.info()

print("\n Info Data Setelah Konversi ke String (selain jumlah dan tahun)")
kolom_string = [
    'id',
    'id_kategori'
]

df2_20232024[kolom_string] = df2_20232024[kolom_string].astype(str)
df2_20232024.info()

# -------------------------
# 2. STANDARDISASI KAB/KOT
# -------------------------
df2_20232024 = df2_20232024[df2_20232024['tahun'].isin([2023, 2024])]
df2_20232024 = df2_20232024[
    df2_20232024['kategori'].isin([
        'tenaga_kesehatan_dokter',
        'tenaga_kesehatan_perawat',
        'tenaga_kesehatan_bidan'
    ])
]

df2_20232024 = df2_20232024.rename(columns={
    'kabupatenkota': 'nama_kabupaten_kota'
})

df2_20232024 = df2_20232024[
    ~df2_20232024['nama_kabupaten_kota']
      .str.strip()
      .str.upper()
      .eq('JAWA TIMUR')
]

df2_20232024['nama_kabupaten_kota'] = (
    df2_20232024['nama_kabupaten_kota']
    .astype(str)
    .str.upper()
    .str.strip()
)

daftar_kabkot = [
    'KABUPATEN PACITAN',
    'KABUPATEN PONOROGO',
    'KABUPATEN TRENGGALEK',
    'KABUPATEN TULUNGAGUNG',
    'KABUPATEN BLITAR',
    'KABUPATEN KEDIRI',
    'KABUPATEN MALANG',
    'KABUPATEN LUMAJANG',
    'KABUPATEN JEMBER',
    'KABUPATEN BANYUWANGI',
    'KABUPATEN BONDOWOSO',
    'KABUPATEN SITUBONDO',
    'KABUPATEN PROBOLINGGO',
    'KABUPATEN PASURUAN',
    'KABUPATEN SIDOARJO',
    'KABUPATEN MOJOKERTO',
    'KABUPATEN JOMBANG',
    'KABUPATEN NGANJUK',
    'KABUPATEN MADIUN',
    'KABUPATEN MAGETAN',
    'KABUPATEN NGAWI',
    'KABUPATEN BOJONEGORO',
    'KABUPATEN TUBAN',
    'KABUPATEN LAMONGAN',
    'KABUPATEN GRESIK',
    'KABUPATEN BANGKALAN',
    'KABUPATEN SAMPANG',
    'KABUPATEN PAMEKASAN',
    'KABUPATEN SUMENEP',
    'KOTA KEDIRI',
    'KOTA BLITAR',
    'KOTA MALANG',
    'KOTA PROBOLINGGO',
    'KOTA PASURUAN',
    'KOTA MOJOKERTO',
    'KOTA MADIUN',
    'KOTA SURABAYA',
    'KOTA BATU'
]

# -------------------------
# 3. CEK KELENGKAPAN KAB/KOT
# -------------------------
tidak_ada = [
nama for nama in daftar_kabkot
if nama not in df2_20232024['nama_kabupaten_kota'].values
]


print("\n2.Cek Kelengkapan Kabupaten/Kota")

if len(tidak_ada) > 0:
    print("Nama Kabupaten/kota yang tidak ada di dataset:")
    for nama in tidak_ada:
        print(nama)
else:
    print("Semua nama kabupaten/kota tersedia di dataset")

# -------------------------
# 4. CEK DUPLIKAT
# -------------------------
jumlah_duplikat = df2_20232024.duplicated().sum()

print("\n3.Cek Data Duplikat")

if jumlah_duplikat > 0:
    print("Jumlah data duplikat:", jumlah_duplikat)
    df2_20232024 = df2_20232024.drop_duplicates()
    print("Berhasil dihapus")
else:
    print("Tidak ada data duplikat")


# -------------------------
# . CEK MISSING VALUE
# -------------------------
missing_value = df2_20232024.isnull().sum()

print("\n4.Cek Missing Value")
for kolom in df2_20232024.columns:
    indeks_nan = df2_20232024[df2_20232024[kolom].isna()].index.tolist()

    if len(indeks_nan) > 0:
        print(f"{kolom}: {indeks_nan}")
        

if missing_value.sum() > 0:
    print("Jumlah missing value:", missing_value.sum())
    kolom_numerik = df2_20232024.select_dtypes(include='number').columns
    for kolom in kolom_numerik:
        df2_20232024[kolom] = df2_20232024[kolom].fillna(df2_20232024[kolom].median())
    print("Berhasil ditangani")

else:
    print("Tidak ada missing value")



# -------------------------
# 6. CEK OUTLIER (IQR)
# -------------------------
hasil_outlier = []
kolom_numerik = df2_20232024.select_dtypes(include='number').columns

print("\n5.Cek Outlier (IQR)")

for kolom in kolom_numerik:

    Q1 = df2_20232024[kolom].quantile(0.25)
    Q3 = df2_20232024[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah_outlier = df2_20232024[
        (df2_20232024[kolom] < batas_bawah) |
        (df2_20232024[kolom] > batas_atas)
    ]


    if len(jumlah_outlier) > 0:
        keterangan = "Ada outlier"
        print(f"\nVariabel: {kolom}")
        print(f'Jumlah Outlier: {len(jumlah_outlier)}')
        display(jumlah_outlier[['nama_kabupaten_kota', 'tahun', 'kategori', kolom]])
    else:
        print(f"\nVariabel: {kolom}")
        print("Tidak ada outlier")


 1. Info Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2028 entries, 0 to 2027
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             2028 non-null   int64 
 1   id_kategori    2028 non-null   int64 
 2   tahun          2028 non-null   int64 
 3   kabupatenkota  2028 non-null   object
 4   kategori       2028 non-null   object
 5   jumlah         2028 non-null   int64 
dtypes: int64(4), object(2)
memory usage: 95.2+ KB

 Info Data Setelah Konversi ke String (selain jumlah dan tahun)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2028 entries, 0 to 2027
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             2028 non-null   object
 1   id_kategori    2028 non-null   object
 2   tahun          2028 non-null   int64 
 3   kabupatenkota  2028 non-null   object
 4   kategori       2028 non-null   object
 5   jumlah

,nama_kabupaten_kota,tahun,kategori,jumlah
185,KABUPATEN SIDOARJO,2024,tenaga_kesehatan_perawat,2203
407,KOTA MALANG,2024,tenaga_kesehatan_perawat,2039
478,KOTA SURABAYA,2024,tenaga_kesehatan_perawat,7446
554,KABUPATEN TULUNGAGUNG,2023,tenaga_kesehatan_perawat,2259
590,KABUPATEN MALANG,2023,tenaga_kesehatan_perawat,3698
612,KABUPATEN JEMBER,2023,tenaga_kesehatan_perawat,2996
632,KABUPATEN BANYUWANGI,2023,tenaga_kesehatan_perawat,2010
698,KABUPATEN SIDOARJO,2023,tenaga_kesehatan_perawat,4111
817,KABUPATEN LAMONGAN,2023,tenaga_kesehatan_perawat,2452
819,KABUPATEN GRESIK,2023,tenaga_kesehatan_perawat,2300


In [5]:
import pandas as pd

# ==============================================================================
# DATA MASTER KODE WILAYAH 
# ==============================================================================

df_master = pd.read_csv('jumlah_tenaga_kesehatan.csv')

df_master['nama_kabupaten_kota'] = (
    df_master['nama_kabupaten_kota']
    .astype(str)
    .str.strip()
    .str.upper()
)

master_kabkot = (
    df_master[['kode_kabupaten_kota', 'nama_kabupaten_kota']]
    .drop_duplicates()
)

master_kabkot

,kode_kabupaten_kota,nama_kabupaten_kota
0,3501,KABUPATEN PACITAN
15,3502,KABUPATEN PONOROGO
30,3503,KABUPATEN TRENGGALEK
45,3504,KABUPATEN TULUNGAGUNG
60,3505,KABUPATEN BLITAR
75,3506,KABUPATEN KEDIRI
90,3507,KABUPATEN MALANG
105,3508,KABUPATEN LUMAJANG
120,3509,KABUPATEN JEMBER
135,3510,KABUPATEN BANYUWANGI


In [6]:
# ==============================================================================
# ISI KODE KABUPATEN/KOTA
# ==============================================================================

df2_20232024 = df2_20232024.drop(
    columns=['kode_kabupaten_kota'],
    errors='ignore'
)

# merge
df2_20232024 = df2_20232024.merge(
    master_kabkot,
    on='nama_kabupaten_kota',
    how='left'
)

df2_20232024

,id,id_kategori,tahun,nama_kabupaten_kota,kategori,jumlah,kode_kabupaten_kota
0,1,522,2024,KABUPATEN PACITAN,tenaga_kesehatan_dokter,0,3501
1,1,522,2024,KABUPATEN PACITAN,tenaga_kesehatan_bidan,3,3501
2,1,522,2024,KABUPATEN PACITAN,tenaga_kesehatan_perawat,185,3501
3,2,522,2024,KABUPATEN PONOROGO,tenaga_kesehatan_dokter,0,3502
4,2,522,2024,KABUPATEN PONOROGO,tenaga_kesehatan_perawat,445,3502
...,...,...,...,...,...,...,...
223,76,522,2023,KOTA SURABAYA,tenaga_kesehatan_dokter,0,3578
224,76,522,2023,KOTA SURABAYA,tenaga_kesehatan_perawat,10883,3578
225,77,522,2023,KOTA BATU,tenaga_kesehatan_perawat,563,3579
226,77,522,2023,KOTA BATU,tenaga_kesehatan_dokter,0,3579


In [7]:
# ==============================================================================
# GABUNG DATA 2018-2022 DAN 2023-2024
# ==============================================================================


df_tenagakesehatan = pd.concat([df1_20182022, df2_20232024], ignore_index=True)

df_tenagakesehatan['kategori'] = df_tenagakesehatan['kategori'].replace({
    'tenaga_kesehatan_dokter': 'DOKTER',
    'tenaga_kesehatan_perawat': 'PERAWAT',
    'tenaga_kesehatan_bidan': 'BIDAN',
    'JUMLAH TENAGA DOKTER UMUM': 'DOKTER',
    'JUMLAH TENAGA PERAWAT': 'PERAWAT',
    'JUMLAH TENAGA BIDAN': 'BIDAN'
})

df_tenagakesehatan['nama_kabupaten_kota'] = (
    df_tenagakesehatan['nama_kabupaten_kota']
    .astype(str)
    .str.upper()
    .str.strip()
)

df_tenagakesehatan = df_tenagakesehatan.sort_values(
    ['tahun', 'nama_kabupaten_kota', 'kategori'],
    ascending=[True, True, True]
).reset_index(drop=True)

df_tenagakesehatan

,id,id_index,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,periode_update,kategori,jumlah,satuan,tahun,id_kategori
0,26,38626,35,JAWA TIMUR,3526,KABUPATEN BANGKALAN,2018-S2,BIDAN,899,ORANG,2018,NaN
1,26,38526,35,JAWA TIMUR,3526,KABUPATEN BANGKALAN,2018-S2,DOKTER,130,ORANG,2018,NaN
2,26,38126,35,JAWA TIMUR,3526,KABUPATEN BANGKALAN,2018-S2,PERAWAT,895,ORANG,2018,NaN
3,10,14910,35,JAWA TIMUR,3510,KABUPATEN BANYUWANGI,2018-S2,BIDAN,953,ORANG,2018,NaN
4,10,13610,35,JAWA TIMUR,3510,KABUPATEN BANYUWANGI,2018-S2,DOKTER,207,ORANG,2018,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
793,33,NaN,NaN,NaN,3574,KOTA PROBOLINGGO,NaN,DOKTER,0,NaN,2024,522
794,33,NaN,NaN,NaN,3574,KOTA PROBOLINGGO,NaN,PERAWAT,278,NaN,2024,522
795,37,NaN,NaN,NaN,3578,KOTA SURABAYA,NaN,BIDAN,69,NaN,2024,522
796,37,NaN,NaN,NaN,3578,KOTA SURABAYA,NaN,DOKTER,0,NaN,2024,522


TRANSFORMASI DATA

In [8]:
# ==============================================================================
# PIVOT JUMLAH TENAGA KESEHATAN PER KABUPATEN/KOTA PER TAHUN
# ==============================================================================

# -------------------------
# PIVOT JUMLAH TENAGA KESEHATAN
# -------------------------
df_tenagakesehatan_tahun_kab = df_tenagakesehatan.pivot_table(
    index=[
        'kode_kabupaten_kota',
        'nama_kabupaten_kota',
        'tahun'
    ],
    columns='kategori',
    values='jumlah',
    aggfunc='sum'
).reset_index()

# Rename kolom hasil pivot
df_tenagakesehatan_tahun_kab = df_tenagakesehatan_tahun_kab.rename(columns={
    'DOKTER': 'JUMLAH_DOKTER',
    'PERAWAT': 'JUMLAH_PERAWAT',
    'BIDAN': 'JUMLAH_BIDAN'
})

df_tenagakesehatan_tahun_kab = df_tenagakesehatan_tahun_kab.sort_values(
    ['tahun', 'kode_kabupaten_kota'],
    ascending=[True, True]
).reset_index(drop=True)

df_tenagakesehatan_tahun_kab

kategori,kode_kabupaten_kota,nama_kabupaten_kota,tahun,JUMLAH_BIDAN,JUMLAH_DOKTER,JUMLAH_PERAWAT
0,3501,KABUPATEN PACITAN,2018,347,141,619
1,3502,KABUPATEN PONOROGO,2018,519,129,990
2,3503,KABUPATEN TRENGGALEK,2018,384,227,792
3,3504,KABUPATEN TULUNGAGUNG,2018,609,263,1098
4,3505,KABUPATEN BLITAR,2018,540,233,837
...,...,...,...,...,...,...
261,3575,KOTA PASURUAN,2024,1,0,261
262,3576,KOTA MOJOKERTO,2024,4,0,396
263,3577,KOTA MADIUN,2024,5,0,521
264,3578,KOTA SURABAYA,2024,69,0,7446


SIMPAN DATA

In [9]:
#===============================================================================
# SIMPAN DATA
#===============================================================================
df_tenagakesehatan_tahun_kab.to_csv('data_jumlah_tenagakesehatan.csv', index=False)

In [10]:
print(df_tenagakesehatan_tahun_kab.columns.tolist())

['kode_kabupaten_kota', 'nama_kabupaten_kota', 'tahun', 'JUMLAH_BIDAN', 'JUMLAH_DOKTER', 'JUMLAH_PERAWAT']
